In [ ]:
import sys
!{sys.executable} -m pip install scikit-learn



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np

red_df = pd.read_csv(r"E:\hw1AlPython\hw05\data\winequality-red.csv", sep=";")
white_df = pd.read_csv(r"E:\hw1AlPython\hw05\data\winequality-white.csv", sep=";")

red_df["is_red"] = 1
white_df["is_red"] = 0

df = pd.concat([red_df, white_df], ignore_index=True)

df.head()


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,is_red
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,1
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,1
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



red_df["is_red"] = 1
white_df["is_red"] = 0

df = pd.concat([red_df, white_df], ignore_index=True)

print(df.isna().sum())
print(df.info())

df["total_acid"] = df["fixed acidity"] + df["volatile acidity"]
df["sugar_to_alc"] = df["residual sugar"] / df["alcohol"]

X = df.drop("quality", axis=1)
y = df["quality"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=11)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=11)

sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_val_sc = sc.transform(X_val)
X_test_sc = sc.transform(X_test)

base_model = LinearRegression()
base_model.fit(X_train_sc, y_train)

v_preds = base_model.predict(X_val_sc)

print("Validation metrics:")
print(f"MAE: {mean_absolute_error(y_val, v_preds)}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, v_preds))}")
print(f"R2: {r2_score(y_val, v_preds)}")

params = {"alpha": [0.01, 0.1, 1, 10, 100]}
search = GridSearchCV(Ridge(), params, scoring="neg_root_mean_squared_error", cv=5)
search.fit(X_train_sc, y_train)

print(f"Best alpha found: {search.best_params_}")

final_model = search.best_estimator_
test_preds = final_model.predict(X_test_sc)

mae_t = mean_absolute_error(y_test, test_preds)
rmse_t = np.sqrt(mean_squared_error(y_test, test_preds))
r2_t = r2_score(y_test, test_preds)

print("\nFinal Test:")
print(mae_t, rmse_t, r2_t)

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
is_red                  0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         6497 non-null   float64
 1   volatile acidity      6497 non-null   float64
 2   citric acid           6497 non-null   float64
 3   residual sugar        6497 non-null   float64
 4   chlorides             6497 non-null   float64
 5   free sulfur dioxide   6497 non-null   float64
 6   total sulfur dioxide  6497 non-null   float64
 7   density               6497 non-null   float64
 8   pH                    6497 